In [12]:
import polars as pl 
import polars_ds as pds 
import requests 
import json 
from utils import scrape_ticket_sections, get_available_events, scrape_match_results, scrape_eliteserien_results,create_table_after_round

In [2]:
available_events = get_available_events()

In [3]:
for event in available_events:
    print(event['event_id'])


1085523
1187151
1188514


In [2]:
eliteserien_results = scrape_eliteserien_results(season_id = 2025,year = 2026)

In [3]:
create_table_after_round(eliteserien_results)

matchday,team,points,goals_for,goals_against,goal_difference,total_points,total_goals_for,total_goals_against,total_goal_difference,table_position
i64,str,i32,i64,i64,i64,i32,i64,i64,i64,u32
1,"""HamKam""",3,2,1,1,3,2,1,1,1
1,"""KFUM Oslo""",3,2,0,2,3,2,0,2,1
1,"""Kristiansund BK""",3,3,2,1,3,3,2,1,1
1,"""Lillestrøm SK""",3,3,1,2,3,3,1,2,1
1,"""Molde FK""",3,2,0,2,3,2,0,2,1
…,…,…,…,…,…,…,…,…,…,…
18,"""KFUM Oslo""",1,1,1,0,19,19,27,-8,12
18,"""Sandefjord""",3,2,1,1,18,15,23,-8,13
18,"""Aalesunds FK""",1,5,5,0,15,27,41,-14,14


In [ ]:
import polars as pl

matches = pl.DataFrame(eliteserien_results)

# Hent målene fra resultatet
matches = matches.with_columns([
    pl.col("result")
      .str.split_exact(":", 1)
      .struct.field("field_0")
      .cast(pl.Int64)
      .alias("home_goals"),

    pl.col("result")
      .str.split_exact(":", 1)
      .struct.field("field_1")
      .cast(pl.Int64)
      .alias("away_goals"),
])

# Lag en rad for hjemmelaget
home = matches.select([
    "matchday",
    pl.col("home_team").alias("team"),
    pl.col("home_goals").alias("goals_for"),
    pl.col("away_goals").alias("goals_against"),
    pl.when(pl.col("home_goals") > pl.col("away_goals"))
      .then(3)
      .when(pl.col("home_goals") == pl.col("away_goals"))
      .then(1)
      .otherwise(0)
      .alias("points"),
])

# Lag en rad for bortelaget
away = matches.select([
    "matchday",
    pl.col("away_team").alias("team"),
    pl.col("away_goals").alias("goals_for"),
    pl.col("home_goals").alias("goals_against"),
    pl.when(pl.col("away_goals") > pl.col("home_goals"))
      .then(3)
      .when(pl.col("away_goals") == pl.col("home_goals"))
      .then(1)
      .otherwise(0)
      .alias("points"),
])

# Slå sammen hjemmelag og bortelag
team_results = pl.concat([home, away])

# Summer resultatene per lag per runde
round_table = (
    team_results
    .group_by(["matchday", "team"])
    .agg([
        pl.col("points").sum(),
        pl.col("goals_for").sum(),
        pl.col("goals_against").sum(),
    ])
    .with_columns(
        (pl.col("goals_for") - pl.col("goals_against"))
        .alias("goal_difference")
    )
    .sort(["team", "matchday"])
)

# Beregn totalsummen etter hver runde
table_after_round = round_table.with_columns([
    pl.col("points")
      .cum_sum()
      .over("team")
      .alias("total_points"),

    pl.col("goals_for")
      .cum_sum()
      .over("team")
      .alias("total_goals_for"),

    pl.col("goals_against")
      .cum_sum()
      .over("team")
      .alias("total_goals_against"),

    pl.col("goal_difference")
      .cum_sum()
      .over("team")
      .alias("total_goal_difference"),
])

# Beregn tabellplassering etter hver runde
table_after_round = (
    table_after_round
    .with_columns(
        pl.col("total_points")
          .rank("min", descending=True)
          .over("matchday")
          .alias("table_position")
    )
    .sort(["matchday", "table_position", "team"])
)


matchday,team,points,goals_for,goals_against,goal_difference,total_points,total_goals_for,total_goals_against,total_goal_difference,table_position
i64,str,i32,i64,i64,i64,i32,i64,i64,i64,u32
1,"""SK Brann""",0,2,3,-1,0,2,3,-1,10
2,"""SK Brann""",0,1,2,-1,0,3,5,-2,14
3,"""SK Brann""",3,5,1,4,3,8,6,2,9
4,"""SK Brann""",0,0,1,-1,3,8,7,1,12
5,"""SK Brann""",0,2,3,-1,3,10,10,0,14
…,…,…,…,…,…,…,…,…,…,…
14,"""SK Brann""",3,2,1,1,16,23,22,1,9
15,"""SK Brann""",0,2,3,-1,16,25,25,0,10
16,"""SK Brann""",3,3,2,1,19,28,27,1,9


In [2]:
season_2026 = scrape_match_results(season_id=2025,year = 2026)

In [4]:
pl.DataFrame(season_2026).sort(by = 'date')

date,home_team,away_team,result,brann_table_position,snapshot_at,brann_goal_scorers
date,str,str,str,i64,"datetime[μs, UTC]",list[str]
2026-01-22,"""SK Brann""","""Midtjylland""","""3:3""",null,2026-08-23 08:53:52.539782 UTC,"[""N. Holm"", ""E. Kornvig"", ""J. Soltvedt""]"
2026-01-29,"""Sturm Graz""","""SK Brann""","""1:0""",null,2026-08-23 08:53:52.539782 UTC,[]
2026-02-19,"""SK Brann""","""Bologna""","""0:1""",null,2026-08-23 08:53:52.539782 UTC,[]
2026-02-26,"""Bologna""","""SK Brann""","""1:0""",null,2026-08-23 08:53:52.539782 UTC,[]
2026-03-08,"""Tromsdalen""","""SK Brann""","""2:3 AET""",null,2026-08-23 08:53:52.539782 UTC,"[""F. Myhre"", ""J. Thorsteinsson"", ""J. Lungi Sørensen""]"
…,…,…,…,…,…,…
2026-07-12,"""SK Brann""","""IK Start""","""2:1""",null,2026-08-23 08:53:52.539782 UTC,"[""N. Holm"", ""K. Eriksen""]"
2026-07-18,"""Molde FK""","""SK Brann""","""1:2""",null,2026-08-23 08:53:52.539782 UTC,"[""D. De Roeve"", ""K. Eriksen""]"
2026-07-26,"""SK Brann""","""Vålerenga""","""2:3""",null,2026-08-23 08:53:52.539782 UTC,"[""N. Castro"", ""F. Myhre""]"


In [4]:
paok = scrape_ticket_sections(1187151)

In [18]:
pl.DataFrame(paok).filter(pl.col('total_seats')==0,pl.col('sold')==0)

snapshot_at,event_id,section_id,section_name,total_seats,sold,available,unavailable,sold_out,other
"datetime[μs, UTC]",i64,i64,str,i64,i64,i64,i64,bool,i64
2026-08-22 20:20:11.449792 UTC,1187151,755042,"""FJORDKRAFT - STÅPLASSER""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755020,"""FJORDKRAFT Felt A""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755046,"""EGD Hjørnet""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755026,"""STORE STÅ NEDRE""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755021,"""FRYDENBØ Felt C Nedre""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755043,"""FRYDENBØ Felt B Nedre""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755038,"""STORE STÅ""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755041,"""Gangen Frydenbø""",0,0,0,0,false,0
